Load Silver + Gold tables

In [0]:
from pyspark.sql import functions as F

dim_facility_df = spark.table(
    "databricks_project1.gold.dim_facility"
)

dim_labor_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

dim_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

fact_df = spark.table(
    "databricks_project1.gold.fact_employee_payroll"
)

print("Dim Facility:", dim_facility_df.count())
print("Dim Labor Position:", dim_labor_df.count())
print("Dim Employee:", dim_employee_df.count())
print("Fact Employee Payroll:", fact_df.count())

In [0]:
print(
    "Silver Employee:",
    spark.table(
        "databricks_project1.silver.employee_payroll"
    ).count()
)

print(
    "Silver Labor Position:",
    spark.table(
        "databricks_project1.silver.labor_position"
    ).count()
)

print(
    "Silver Facility:",
    spark.table(
        "databricks_project1.silver.facility"
    ).count()
)

In [0]:
from pyspark.sql import functions as F

employee_silver_df = spark.table(
    "databricks_project1.silver.employee_payroll"
)

dim_employee_df = spark.table(
    "databricks_project1.gold.dim_employee"
)

dim_facility_df = spark.table(
    "databricks_project1.gold.dim_facility"
)

dim_labor_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

print("Silver Employee:", employee_silver_df.count())
print("Dim Employee:", dim_employee_df.count())
print("Dim Facility:", dim_facility_df.count())
print("Dim Labor Position:", dim_labor_df.count())

## Prepare the current employee dimension
## 
Because DimEmployee is SCD Type 2, we must only join against the current employee record.

In [0]:
dim_employee_current_df = (
    dim_employee_df
    .filter(F.col("IsCurrent") == True)
    .select(
        "EmployeeKey",
        "Employee_Code"
    )
)

print(
    f"Current DimEmployee records: "
    f"{dim_employee_current_df.count()}"
)

## Prepare Facility dimension

In [0]:
dim_facility_lookup_df = (
    dim_facility_df
    .select(
        F.col("Facility_Code").alias("Dim_Facility_Code"),
        F.col("Facility_Name").alias("Dim_Facility_Name")
    )
    .dropDuplicates(["Dim_Facility_Code"])
)

## Prepare Labor Position dimension

In [0]:
dim_labor_lookup_df = (
    dim_labor_df
    .select(
        F.col("Labor_Position_Code").alias(
            "Dim_Labor_Position_Code"
        ),
        F.col("Labor_Position_Desc").alias(
            "Dim_Labor_Position_Desc"
        )
    )
    .dropDuplicates(["Dim_Labor_Position_Code"])
)

## Build the Fact table
## 
The important part here is that the fact table gets the surrogate EmployeeKey from DimEmployee.

In [0]:
# Temp cell---------------------------------------------------------
# Prepare CURRENT Employee Dimension
# Keep exactly ONE current record per Employee_Code
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

dim_employee_source_df = (
    spark.table(
        "databricks_project1.gold.dim_employee"
    )
    .filter(
        F.col("IsCurrent") == True
    )
)

# If multiple current rows exist for an Employee_Code,
# retain the latest one using StartDate.
dim_employee_window = (
    Window
    .partitionBy("Employee_Code")
    .orderBy(
        F.col("StartDate").desc(),
        F.col("EmployeeKey").desc()
    )
)

dim_employee_current_df = (
    dim_employee_source_df
    .withColumn(
        "_rn",
        F.row_number().over(dim_employee_window)
    )
    .filter(
        F.col("_rn") == 1
    )
    .drop("_rn")
)

print(
    "DimEmployee current records:",
    dim_employee_current_df.count()
)

print(
    "Distinct current Employee_Code:",
    dim_employee_current_df
    .select("Employee_Code")
    .distinct()
    .count()
)

duplicate_dim_employee_df = (
    dim_employee_current_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate current Employee_Code:",
    duplicate_dim_employee_df.count()
)

display(duplicate_dim_employee_df)

In [0]:
# ---------------------------------------------------------
# Prepare source Employee records
# Keep only the latest record for each Employee_Code
# ---------------------------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

employee_source_df = (
    employee_silver_df
    .withColumn(
        "_ingestion_ts",
        F.to_timestamp("ingestion_timestamp")
    )
)

employee_window = (
    Window
    .partitionBy("Employee_Code")
    .orderBy(F.col("_ingestion_ts").desc())
)

employee_latest_df = (
    employee_source_df
    .withColumn(
        "_rn",
        F.row_number().over(employee_window)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_ingestion_ts")
)

print(
    "Silver Employee records:",
    employee_silver_df.count()
)

print(
    "Latest unique Employee records:",
    employee_latest_df.count()
)

In [0]:
# ---------------------------------------------------------
# Build Fact Employee Payroll
# ---------------------------------------------------------

fact_employee_payroll_df = (
    employee_latest_df.alias("e")

    # Current Employee Dimension
    .join(
        dim_employee_current_df.alias("de"),
        F.col("e.Employee_Code") ==
        F.col("de.Employee_Code"),
        "inner"
    )

    # Facility lookup
    .join(
        dim_facility_lookup_df.alias("df"),
        F.col("e.Facility_Code") ==
        F.col("df.Dim_Facility_Code"),
        "left"
    )

    # Labor Position lookup
    .join(
        dim_labor_lookup_df.alias("dl"),
        F.col("e.Labor_Position_Code") ==
        F.col("dl.Dim_Labor_Position_Code"),
        "left"
    )

    .select(
        F.col("de.EmployeeKey"),
        F.col("e.Employee_Code"),
        F.col("e.Employee_Status"),
        F.col("e.Facility_Code"),
        F.col("df.Dim_Facility_Name").alias("Facility_Name"),
        F.col("e.Labor_Position_Code"),
        F.col("dl.Dim_Labor_Position_Desc").alias(
            "Labor_Position_Desc"
        ),
        F.col("e.DOB").alias("Birth_Date"),
        F.col("e.Hire_Date"),
        F.col("e.Rehire_Date"),
        F.col("e.Termination_Date"),
        F.col("e.ingestion_timestamp"),
        F.col("e.source_file")
    )
)

print(
    "Fact source records:",
    fact_employee_payroll_df.count()
)

the duplicate validation cell

In [0]:
# ---------------------------------------------------------
# Validate duplicate Employee_Code
# ---------------------------------------------------------

duplicate_fact_df = (
    fact_employee_payroll_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_fact_df.count()

print(
    "Duplicate Employee_Code in Fact source:",
    duplicate_count
)

display(duplicate_fact_df)

the duplicate validation cell

Validate employee joins

In [0]:
missing_employee_df = fact_employee_payroll_df.filter(
    F.col("EmployeeKey").isNull()
)

print(
    f"Employees without DimEmployee match: "
    f"{missing_employee_df.count()}"
)

Validate Facility joins

In [0]:
missing_facility_df = fact_employee_payroll_df.filter(
    F.col("Facility_Name").isNull()
)

print(
    f"Employees without Facility match: "
    f"{missing_facility_df.count()}"
)



Validate Labor Position joins

In [0]:
missing_labor_df = fact_employee_payroll_df.filter(
    F.col("Labor_Position_Desc").isNull()
)

print(
    f"Employees without Labor Position match: "
    f"{missing_labor_df.count()}"
)

In [0]:
# temp cell---------------------------------------------------------
# Identify missing Labor Position codes
# ---------------------------------------------------------

missing_labor_codes_df = (
    employee_latest_df.alias("e")
    .join(
        dim_labor_lookup_df.alias("dl"),
        F.col("e.Labor_Position_Code") ==
        F.col("dl.Dim_Labor_Position_Code"),
        "left_anti"
    )
    .select("Labor_Position_Code")
    .distinct()
    .orderBy("Labor_Position_Code")
)

print(
    "Missing distinct Labor Position codes:",
    missing_labor_codes_df.count()
)

display(missing_labor_codes_df)

Check duplicate employee facts

In [0]:
duplicate_fact_df = (
    fact_employee_payroll_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicate Employee_Code in fact: "
    f"{duplicate_fact_df.count()}"
)

display(duplicate_fact_df)

In [0]:
missing_labor_codes_df = (
    employee_silver_df
    .select("Labor_Position_Code")
    .distinct()
    .join(
        dim_labor_df.select("Labor_Position_Code").distinct(),
        on="Labor_Position_Code",
        how="left_anti"
    )
    .orderBy("Labor_Position_Code")
)

print(
    f"Missing Labor Position codes: "
    f"{missing_labor_codes_df.count()}"
)

display(missing_labor_codes_df)

In [0]:
missing_labor_employee_df = (
    employee_silver_df.alias("e")
    .join(
        missing_labor_codes_df.alias("m"),
        F.col("e.Labor_Position_Code") ==
        F.col("m.Labor_Position_Code"),
        "inner"
    )
    .select(
        F.col("e.Employee_Code").alias("Employee_Code"),
        F.col("e.Labor_Position_Code").alias("Labor_Position_Code")
    )
)

print(
    f"Employee records with missing Labor Position: "
    f"{missing_labor_employee_df.count()}"
)

display(
    missing_labor_employee_df
    .orderBy("Labor_Position_Code", "Employee_Code")
)

In [0]:
display(
    dim_labor_df
    .filter(
        F.col("Labor_Position_Code").isin(5502, 9510)
    )
)


In [0]:
display(
    employee_silver_df
    .filter(
        F.col("Labor_Position_Code").isin(5502, 9510)
    )
    .select(
        "Employee_Code",
        "Labor_Position_Code",
        "Employee_Status",
        "Facility_Code"
    )
    .orderBy("Labor_Position_Code")
)

In [0]:
from pyspark.sql import functions as F

labor_silver_df = spark.table(
    "databricks_project1.silver.labor_position"
)

print("Silver Labor Position count:", labor_silver_df.count())

display(
    labor_silver_df.filter(
        F.col("Labor_Position_Code").isin(5502, 9510)
    )
)

In [0]:
from pyspark.sql import functions as F

employee_silver_df = spark.table(
    "databricks_project1.silver.employee_payroll"
)

labor_gold_df = spark.table(
    "databricks_project1.gold.dim_labor_position"
)

fresh_missing_labor_df = (
    employee_silver_df.alias("e")
    .join(
        labor_gold_df
        .select("Labor_Position_Code")
        .alias("l"),
        F.col("e.Labor_Position_Code") ==
        F.col("l.Labor_Position_Code"),
        "left_anti"
    )
    .select(
        F.col("e.Employee_Code"),
        F.col("e.Labor_Position_Code")
    )
)

print(
    "Current unmatched employees:",
    fresh_missing_labor_df.count()
)

display(fresh_missing_labor_df)

In [0]:
print(
    "Final Fact EmployeePayroll count:",
    fact_employee_payroll_df.count()
)

display(
    fact_employee_payroll_df
    .filter(F.col("Employee_Code") == "ALWL")
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

FACT_TABLE = "databricks_project1.gold.fact_employee_payroll"

# Existing Fact table
fact_delta = DeltaTable.forName(
    spark,
    FACT_TABLE
)

# MERGE source into existing Fact
(
    fact_delta.alias("tgt")
    .merge(
        fact_employee_payroll_df.alias("src"),
        "tgt.Employee_Code = src.Employee_Code"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Fact EmployeePayroll incremental MERGE completed successfully")

In [0]:
fact_check_df = spark.table(
    "databricks_project1.gold.fact_employee_payroll"
)

print(
    "Fact EmployeePayroll count:",
    fact_check_df.count()
)

In [0]:
duplicate_fact_check_df = (
    fact_check_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate Employee_Code in Fact:",
    duplicate_fact_check_df.count()
)

display(duplicate_fact_check_df)

In [0]:
print(
    "Final Fact EmployeePayroll count:",
    fact_employee_payroll_df.count()
)

In [0]:
fact_final_df = spark.table(
    "databricks_project1.gold.fact_employee_payroll"
)

print(
    "Final Fact EmployeePayroll count:",
    fact_final_df.count()
)

In [0]:
display(
    fact_final_df
    .groupBy("Employee_Code")
    .count()
    .filter(F.col("count") > 1)
)